In [1]:
from model_zoo import (
    ModelZoo, ModelRegistry, MetricRegistry,
    ModelSpec, ModelKind, EvalMetricSpec, MetricKind, MetricDirection,
    HyperParamSpace, HyperParamField, DType,
    ExecutionBinding, BindingKind,
    DataRef, DataRefType, ExecutionContext,
    ModelRunRequest, MetricComputeRequest, EvaluationSpec,
)
from model_zoo.runners.library import LibraryModelRunner, LibraryMetricRunner
from model_zoo.examples.models import make_synthetic_linear, make_synthetic_mtcars_wt_mpg, make_synthetic_mtcars_multi
from model_zoo import r_style_summary_string
from model_zoo.utils.logging import configure_logging, get_logger, quiet_third_party_loggers
from model_zoo.examples.sklearn_models import sk_lr_train_and_infer  # noqa: F401 (for binding importability)
from model_zoo.examples.pymc_models import pymc_linear_nuts  # noqa: F401 (for binding importability)
from model_zoo.examples import metrics as example_metrics
from model_zoo.contracts.validation import ParamValidationService
from model_zoo import DefaultParamValidator
from model_zoo.contracts.audit import AuditLog, AuditEvent

class PrintAudit(AuditLog):
    def append(self, event: AuditEvent) -> None:
        print(f"[AUDIT] {event.kind} ref={event.ref_id}")

In [2]:
class NoOpValidator(ParamValidationService):
    def validate_model_params(self, model_spec, params): return None
    def validate_metric_params(self, metric_spec, params): return None

class PrintAudit(AuditLog):
    def append(self, event: AuditEvent) -> None:
        print(f"[AUDIT] {event.kind} ref={event.ref_id}")

In [3]:
model_reg, metric_reg = ModelRegistry(), MetricRegistry()
zoo = ModelZoo(model_reg, metric_reg,
                   binding_registry=None,  # not needed; we pass binding on request in this demo
                   model_runner=LibraryModelRunner(),
                   metric_runner=LibraryMetricRunner(),
                   validator=DefaultParamValidator(),
                   audit=PrintAudit())

In [4]:
lin_hp = HyperParamSpace(fields={
        "fit_intercept": HyperParamField("fit_intercept", DType.BOOL, default=True, required=False,
                                          description="Include intercept in linear model"),
    })

bayes_hp = HyperParamSpace(fields={
        "sigma": HyperParamField("sigma", DType.FLOAT, default=0.5, required=True,
                                  description="Observation noise std"),
        "prior_tau": HyperParamField("prior_tau", DType.FLOAT, default=10.0, required=False,
                                      description="Prior std for slope"),
        "fit_intercept": HyperParamField("fit_intercept", DType.BOOL, default=True, required=False),
    })

In [5]:
lin_spec = ModelSpec(
        model_id="linear_regression_v1",
        name="LinearRegression_OLS",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=lin_hp,
        tags=["linear", "ols"],
        metadata={"example": True}
    )

bayes_spec = ModelSpec(
        model_id="bayes_linear_v1",
        name="BayesianLinearRegression_Conjugate",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=bayes_hp,
        tags=["bayesian", "regression"],
        metadata={"example": True}
    )

In [6]:
zoo.register_model(lin_spec, actor="you")
zoo.register_model(bayes_spec, actor="you")

[AUDIT] REGISTER_MODEL ref=linear_regression_v1
[AUDIT] REGISTER_MODEL ref=bayes_linear_v1


In [7]:
r2_spec = EvalMetricSpec(
        metric_id="r2_v1",
        name="R2_Score",
        version="1.0.0",
        direction=MetricDirection.HIGHER_IS_BETTER,
        kind=MetricKind.LIBRARY,
        tags=["regression", "goodness_of_fit"],
    )
mse_spec = EvalMetricSpec(
        metric_id="mse_v1",
        name="MeanSquaredError",
        version="1.0.0",
        direction=MetricDirection.LOWER_IS_BETTER,
        kind=MetricKind.LIBRARY,
        tags=["regression", "error"],
    )
zoo.register_metric(r2_spec, actor="you")
zoo.register_metric(mse_spec, actor="you")

[AUDIT] REGISTER_METRIC ref=r2_v1
[AUDIT] REGISTER_METRIC ref=mse_v1


In [8]:
# List models
print("Registered models:")
for m in zoo.list_models():
        print(" -", m.model_id, m.name, m.version)
print("Registered metrics:")
for mt in zoo.list_metrics():
        print(" -", mt.metric_id, mt.name, mt.version)

Registered models:
 - linear_regression_v1 LinearRegression_OLS 1.0.0
 - bayes_linear_v1 BayesianLinearRegression_Conjugate 1.0.0
Registered metrics:
 - r2_v1 R2_Score 1.0.0
 - mse_v1 MeanSquaredError 1.0.0


In [9]:
# Prepare synthetic dataset (INLINE DataRefs)
X, y = make_synthetic_linear(n=100, slope=2.0, intercept=1.0, noise_sigma=0.5)
X_ref = DataRef(ref_type=DataRefType.INLINE, inline=X)
y_ref = DataRef(ref_type=DataRefType.INLINE, inline=y)

# Train/test split indices (80/20)
n = len(X)
split = int(0.8 * n)
train_idx = list(range(0, split))
test_idx = list(range(split, n))


In [10]:
lin_binding = ExecutionBinding(
        bound_id="linear_regression_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.models", callable="linreg_train_and_infer",
    )
bayes_binding = ExecutionBinding(
        bound_id="bayes_linear_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.models", callable="bayes_linreg_train_and_infer",
    )

In [11]:
r2_binding = ExecutionBinding(
        bound_id="r2_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.metrics", callable="r2_score",
    )
mse_binding = ExecutionBinding(
        bound_id="mse_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.metrics", callable="mse",
    )

In [12]:
# Run linear regression with training on train_idx and evaluate on test_idx
lin_req = ModelRunRequest(
        model_id="linear_regression_v1",
        inputs={"X": X_ref, "y": y_ref},
        params={"fit_intercept": True, "train_indices": train_idx},
        context=ExecutionContext(random_seed=2025),
        dataset_version="synthetic:v1",
        binding=lin_binding,
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            test_indices=test_idx,
            bindings={
                "r2_v1": r2_binding,
                "mse_v1": mse_binding,
            }
        ),
    )

In [41]:
def wait_for_result(zoo_obj, run_id: str, actor: str = "you", timeout_s: float = 300.0):
        import time
        start = time.monotonic()
        sleep = 0.05
        last = None
        while True:
            res = zoo_obj.get_run_result(run_id, actor=actor)
            last = res
            # Break when terminal status or in_progress flag cleared
            if res.status.name in ("SUCCEEDED", "FAILED", "CANCELLED"):
                return res
            if not res.metadata.get("in_progress"):
                return res
            if time.monotonic() - start > timeout_s:
                return res
            time.sleep(sleep)
            sleep = min(sleep * 1.5, 1.0)
        return last

In [14]:
lin_handle = zoo.submit_run(lin_req, actor="you")

[AUDIT] RUN_SUBMIT ref=linear_regression_v1


In [15]:
try:
        lin_handle = zoo.submit_run(lin_req, actor="you")
        lin_res = wait_for_result(zoo, lin_handle.run_id, actor="you")
except Exception:
        log.exception("Linear run failed")
        raise
print("LinearRegression: status=", lin_res.status.value)
print(r_style_summary_string(lin_res.summary))

[AUDIT] RUN_SUBMIT ref=linear_regression_v1
[AUDIT] RUN_RESULT ref=run_20251009_111003_935720
[AUDIT] RUN_RESULT ref=run_20251009_111003_935720
LinearRegression: status= succeeded
Call:
  model_zoo.examples.models::linreg_train_and_infer(
    fit_intercept = True
    train_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
)

Coefficients:
               Estimate
  -----------  --------
  coef[x]      1.982883
  intercept    1.143215

  R-squared: 0.993351, MSE: 0.218387
  n=100, n_valid=20

Train metrics:
  r2_v1: 0.993351
  mse_v1: 0.218387

Validation metrics:
  r2_v1: 0.887214
  mse_v1: 0.200727


In [16]:
bayes_req = ModelRunRequest(
        model_id="bayes_linear_v1",
        inputs={"X": X_ref, "y": y_ref},
        params={
            "sigma": 0.5,
            "prior_mu": 0.0,
            "prior_tau": 5.0,
            "fit_intercept": True,
            "train_indices": train_idx,
            "draws": 200,
            "tune": 100,
            "priors": {"beta": {"mu": 0.0, "tau": 5.0}},
        },
        context=ExecutionContext(random_seed=2025),
        dataset_version="synthetic:v1",
        binding=bayes_binding,
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            compute_train=True,
            compute_valid=True,
            train_indices=train_idx,
            test_indices=test_idx,
            bindings={
                "r2_v1": r2_binding,
                "mse_v1": mse_binding,
            }
        ),
    )
try:
        bayes_handle = zoo.submit_run(bayes_req, actor="you")
        bayes_res = wait_for_result(zoo, bayes_handle.run_id, actor="you")
except Exception:
        log.exception("Bayesian run failed")
        raise
print("BayesLinear: status=", bayes_res.status.value)
print(r_style_summary_string(bayes_res.summary))

    # Run Bayesian model again showing explicit priors configuration in hyperparameters
    # This time include a Gamma prior on sigma (precision conjugacy via Normal-Gamma)
custom_priors = {
        "beta": {"distribution": "normal", "mu": 0.5, "tau": 2.0},
        "sigma": {"distribution": "gamma", "shape": 3.0, "rate": 2.0},
    }
bayes_req_custom = ModelRunRequest(
        model_id="bayes_linear_v1",
        inputs={"X": X_ref, "y": y_ref},
        params={
            "sigma": 0.5,
            "prior_mu": 0.0,      # can be omitted when using 'priors'
            "prior_tau": 10.0,    # can be omitted when using 'priors'
            "fit_intercept": True,
            "train_indices": train_idx,
            "draws": 100,
            "priors": custom_priors,
        },
        context=ExecutionContext(random_seed=2025),
        dataset_version="synthetic:v1",
        binding=bayes_binding,
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            compute_train=True,
            compute_valid=True,
            train_indices=train_idx,
            test_indices=test_idx,
            bindings={
                "r2_v1": r2_binding,
                "mse_v1": mse_binding,
            }
        ),
    )
try:
        bayes_handle2 = zoo.submit_run(bayes_req_custom, actor="you")
        bayes_res2 = wait_for_result(zoo, bayes_handle2.run_id, actor="you")
except Exception:
        log.exception("Bayesian run (custom priors) failed")
        raise
print("BayesLinear (custom priors): status=", bayes_res2.status.value)
print(r_style_summary_string(bayes_res2.summary))

[AUDIT] RUN_SUBMIT ref=bayes_linear_v1
[AUDIT] RUN_RESULT ref=run_20251009_111003_954874
[AUDIT] RUN_RESULT ref=run_20251009_111003_954874
BayesLinear: status= succeeded
Call:
  model_zoo.examples.models::bayes_linreg_train_and_infer(
    sigma = 0.5
    prior_mu = 0.0
    prior_tau = 5.0
    fit_intercept = True
    train_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
    draws = 200
    tune = 100
    priors = {'beta': {'mu': 0.0, 'tau': 5.0}}
)

Coefficients:
                       Estimate
  -------------------  --------
  coef[x]              1.982837
  intercept            1.143403
  posterior_beta_mean  1.982837
  posterior_beta_var   0.000586
  sigma                0.500000

  R-squared: 0.989485, MSE: 0.2228

In [17]:
sk_spec = ModelSpec(
        model_id="sklearn_linear_v1",
        name="SklearnLinearRegression",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=HyperParamSpace(fields={
            "fit_intercept": HyperParamField("fit_intercept", DType.BOOL, default=True),
        }),
        tags=["sklearn", "linear"],
        metadata={"example": True}
    )
zoo.register_model(sk_spec, actor="you")
sk_binding = ExecutionBinding(
        bound_id="sklearn_linear_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.sklearn_models", callable="sk_lr_train_and_infer",
    )
sk_req = ModelRunRequest(
        model_id="sklearn_linear_v1",
        inputs={"X": X_ref, "y": y_ref},
        params={"fit_intercept": True, "train_indices": train_idx},
        binding=sk_binding,
        context=ExecutionContext(random_seed=2025),
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            compute_train=True,
            compute_valid=True,
            train_indices=train_idx,
            test_indices=test_idx,
            bindings={"r2_v1": r2_binding, "mse_v1": mse_binding},
        ),
    )


[AUDIT] REGISTER_MODEL ref=sklearn_linear_v1


In [18]:
try:
        sk_handle = zoo.submit_run(sk_req, actor="you")
        sk_res = wait_for_result(zoo, sk_handle.run_id, actor="you")
        print("SklearnLinear: status=", sk_res.status.value)
        print(r_style_summary_string(sk_res.summary))
except Exception as e:
        print("Sklearn example skipped: %s", e)

[AUDIT] RUN_SUBMIT ref=sklearn_linear_v1
[AUDIT] RUN_RESULT ref=run_20251009_111003_991283
[AUDIT] RUN_RESULT ref=run_20251009_111003_991283
SklearnLinear: status= succeeded
Call:
  model_zoo.examples.sklearn_models::sk_lr_train_and_infer(
    fit_intercept = True
    train_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
)

Coefficients:
               Estimate
  -----------  --------
  coef[x]      1.982883
  intercept    1.143215

  R-squared: 0.989485, MSE: 0.222802
  n=100, n_train=80, n_valid=20

Train metrics:
  r2_v1: 0.989485
  mse_v1: 0.222802

Validation metrics:
  r2_v1: 0.887214
  mse_v1: 0.200727


In [19]:
# Multivariate mtcars-like: mpg ~ wt + hp + disp
X_rows, y_m, feat_names = make_synthetic_mtcars_multi()
X_rows_ref = DataRef(ref_type=DataRefType.INLINE, inline=X_rows)
y_m_ref = DataRef(ref_type=DataRefType.INLINE, inline=y_m)
n_mv = len(X_rows)
split_mv = int(0.75 * n_mv)
tr_idx_mv = list(range(split_mv))
te_idx_mv = list(range(split_mv, n_mv))

    # Sklearn multivariate
sk_multi_spec = ModelSpec(
        model_id="sklearn_linear_mtcars_v1",
        name="SklearnLinearRegression_MTCarsMulti",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=HyperParamSpace(fields={
            "fit_intercept": HyperParamField("fit_intercept", DType.BOOL, default=True),
            "feature_names": HyperParamField("feature_names", DType.LIST, default=None),
        }),
        tags=["sklearn", "linear", "mtcars"],
        metadata={"example": True}
    )
zoo.register_model(sk_multi_spec, actor="you")
sk_multi_binding = ExecutionBinding(
        bound_id="sklearn_linear_mtcars_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.sklearn_models", callable="sk_lr_train_and_infer",
    )
sk_multi_req = ModelRunRequest(
        model_id="sklearn_linear_mtcars_v1",
        inputs={"X": X_rows_ref, "y": y_m_ref},
        params={"fit_intercept": True, "train_indices": tr_idx_mv, "feature_names": feat_names},
        binding=sk_multi_binding,
        context=ExecutionContext(random_seed=2025),
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            compute_train=True,
            compute_valid=True,
            train_indices=tr_idx_mv,
            test_indices=te_idx_mv,
            bindings={"r2_v1": r2_binding, "mse_v1": mse_binding},
        ),
    )
try:
        skm_handle = zoo.submit_run(sk_multi_req, actor="you")
        skm_res = wait_for_result(zoo, skm_handle.run_id, actor="you")
        print("SklearnLinear (mtcars multi): status=", skm_res.status.value)
        print(r_style_summary_string(skm_res.summary))
except Exception as e:
        print("Sklearn multivariate example skipped: %s", e)

[AUDIT] REGISTER_MODEL ref=sklearn_linear_mtcars_v1
[AUDIT] RUN_SUBMIT ref=sklearn_linear_mtcars_v1
[AUDIT] RUN_RESULT ref=run_20251009_111004_725209
[AUDIT] RUN_RESULT ref=run_20251009_111004_725209
SklearnLinear (mtcars multi): status= succeeded
Call:
  model_zoo.examples.sklearn_models::sk_lr_train_and_infer(
    fit_intercept = True
    train_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
    feature_names = ['wt', 'hp', 'disp']
)

Coefficients:
                Estimate
  -----------  ---------
  coef[wt]     -4.463286
  coef[hp]     -0.000572
  coef[disp]   -0.008789
  intercept    40.955647

  R-squared: 0.880618, MSE: 2.853203
  n=32, n_train=24, n_valid=8

Train metrics:
  r2_v1: 0.880618
  mse_v1: 2.853203

Validation metrics:
  r2_v1: 0.567410
  mse_v1: 8.709101


In [20]:
pymc_multi_spec = ModelSpec(
        model_id="pymc_linear_nuts_mtcars_v1",
        name="PyMCLinearNUTS_MTCarsMulti",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=HyperParamSpace(fields={
            "draws": HyperParamField("draws", DType.INT, default=500),
            "tune": HyperParamField("tune", DType.INT, default=500),
            "target_accept": HyperParamField("target_accept", DType.FLOAT, default=0.9),
            "priors": HyperParamField("priors", DType.DICT, default=None),
            "feature_names": HyperParamField("feature_names", DType.LIST, default=None),
        }),
        tags=["pymc", "nuts", "bayesian", "mtcars"],
        metadata={"example": True}
    )
zoo.register_model(pymc_multi_spec, actor="you")
pymc_multi_binding = ExecutionBinding(
        bound_id="pymc_linear_nuts_mtcars_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.pymc_models", callable="pymc_linear_nuts",
    )

[AUDIT] REGISTER_MODEL ref=pymc_linear_nuts_mtcars_v1


In [42]:

pymc_multi_req = ModelRunRequest(
        model_id="pymc_linear_nuts_mtcars_v1",
        inputs={"X": X_rows_ref, "y": y_m_ref},
        params={
            "draws": 300,
            "tune": 300,
            "target_accept": 0.9,
            "chains": 5,
            "cores": 1,
            "compute_convergence_checks": True,
            "train_indices": tr_idx_mv,
            "feature_names": feat_names,
            # scalar priors per-feature (shared), but can also pass lists matching feat_names
            "priors": {"beta": {"mu": 0.0, "tau": 10.0}, "sigma": {"scale": 2.0}},
        },
        binding=pymc_multi_binding,
        context=ExecutionContext(random_seed=2025),
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            compute_train=True,
            compute_valid=True,
            train_indices=tr_idx_mv,
            test_indices=te_idx_mv,
            bindings={"r2_v1": r2_binding, "mse_v1": mse_binding},
        ),
    )
try:
        pmm_handle = zoo.submit_run(pymc_multi_req, actor="you")
        pmm_res = wait_for_result(zoo, pmm_handle.run_id, actor="you")
        print("PyMC NUTS (mtcars multi): status=", pmm_res.status.value)
        print(r_style_summary_string(pmm_res.summary))
except Exception as e:
        print("PyMC multivariate example skipped: %s", e)

Initializing NUTS using jitter+adapt_diag...


[AUDIT] RUN_SUBMIT ref=pymc_linear_nuts_mtcars_v1
[AUDIT] RUN_RESULT ref=run_20251009_111345_810363


Sequential sampling (5 chains in 1 job)
NUTS: [alpha, beta, sigma]
Sampling 5 chains for 300 tune and 300 draw iterations (1_500 + 1_500 draws total) took 4 seconds.


[AUDIT] RUN_RESULT ref=run_20251009_111345_810363
PyMC NUTS (mtcars multi): status= succeeded
Call:
  model_zoo.examples.pymc_models::pymc_linear_nuts(
    draws = 300
    tune = 300
    target_accept = 0.9
    chains = 5
    cores = 1
    compute_convergence_checks = True
    train_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
    feature_names = ['wt', 'hp', 'disp']
    priors = {'beta': {'mu': 0.0, 'tau': 10.0}, 'sigma': {'scale': 2.0}}
)

Coefficients:
                   Estimate
  -------------  ----------
  alpha           39.380873
  draws          300.000000
  tune           300.000000
  target_accept    0.900000
  chains           5.000000
  coef[wt]        -4.210166
  coef[hp]         0.001609
  coef[disp]      -0.007801

  R-squared: 0.876504, MSE: 2.951524
  n=32, n_train=24, n_valid=8

Train metrics:
  r2_v1: 0.876504
  mse_v1: 2.951524

Validation metrics:
  r2_v1: 0.575912
  mse_v1: 8.537948


In [40]:
print("PyMC NUTS (mtcars multi): status=", pmm_res.status.value)

print(r_style_summary_string(pmm_res.summary))


PyMC NUTS (mtcars multi): status= started
<no summary available>
